# Combined Propagation + Klein-Gordon Methods

This notebook is the unified entry point for running both:
- real-space propagation methods (Fresnel, Angular Spectrum, WPM)
- Fourier-space forward Klein-Gordon Lanczos propagation


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from wide_angle_propagation.propagation_methods import (
    simulate_fresnel_as,
    simulate_wpm,
    fresnel_propagation_kernel,
    angular_spectrum_propagation_kernel,
    energy2wavelength,
    electron_refractive_index,
)
from wide_angle_propagation.propagation_methods import (
    beam_amplitudes_fwd_direct_allbeams,
)


In [ ]:
# Minimal shared setup for quick method checks
ENERGY = 300e3
GPTS = (64, 64)
SAMPLING = (0.1, 0.1)
DZ = 2.0
N_SLICES = 2

potential = jnp.zeros((N_SLICES, *GPTS), dtype=jnp.float64)
probe = jnp.ones(GPTS, dtype=jnp.complex128)

fk = fresnel_propagation_kernel(GPTS[0], GPTS[1], SAMPLING, z=DZ, energy=ENERGY)
ak = angular_spectrum_propagation_kernel(GPTS[0], GPTS[1], SAMPLING, z=DZ, energy=ENERGY)

exit_fresnel, _, _ = simulate_fresnel_as(potential, probe, fk, DZ, ENERGY)
exit_wpm, _, _ = simulate_wpm(potential, probe, DZ, ENERGY, SAMPLING)

kg_amps, _, _ = beam_amplitudes_fwd_direct_allbeams(
    potential=potential,
    slice_thickness=DZ,
    energy=ENERGY,
    sampling=SAMPLING,
    n_cells_array=np.array([0, 1]),
    gpts=GPTS,
    lanczos_m=40,
)

print("Fresnel |psi| mean:", float(jnp.mean(jnp.abs(exit_fresnel))))
print("WPM |psi| mean:", float(jnp.mean(jnp.abs(exit_wpm))))
print("KG (0,0) amplitudes:", np.asarray(kg_amps[(0, 0)]))
